In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install langchain-google-genai;
!pip install -U langchain-community;
!pip install faiss-cpu;

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
!pip install PyPDF2
from PyPDF2 import PdfReader
from langchain_google_genai import ChatGoogleGenerativeAI

def extract_text_from_pdf(file_path):
    text = ""
    with open(file_path, 'rb') as f:
        reader = PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

**LONG CONTEXT:**
*Gemini has long-context window but I am using chunking just incase the research paper is too long*

In [ ]:
#Chunking: Breaking down complex info into smaller chunks to improve memory
#Max number of tokens (words) is 1500
def split_into_chunks(text, max_tokens=1500):
    #Simple split by paragraphs
    paragraphs = text.split("\n\n")
    chunks, current_chunk = [], ""
    for p in paragraphs:
        if len(current_chunk) + len(p) < max_tokens: #Create chunks
            current_chunk += p + "\n\n"
        else:
            chunks.append(current_chunk)
            current_chunk = p
    chunks.append(current_chunk)
    return chunks


In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

def ask_question(question, context):
    prompt = f"""You are a research assistant. Based on the paper content below, answer the question.
    
Paper content:
{context}

Question: {question}
"""
    response = model.invoke(prompt)
    return response.text


In [ ]:
def summarize_paper(paper_text):
    prompt = (
        "You are a research assistant. Provide a brief summary of this paper in 5 bullet points.\n"
        "Then, create a table of contents with section titles and approximate page numbers.\n\n"
        f"{paper_text[:10000]}"  # Trim for performance 10000 words
    )
    response = model.invoke(prompt)
    return response.content


In [ ]:
paper_text = extract_text_from_pdf("/kaggle/input/sdnfloodlight/ExperimentingwithScalabilityofFloodlightControllerinSoftwareDefinedNetworks.pdf")

#Retrieval-Augmented Generation (RAG):
1. Split the paper into chunks.
2. Embed each chunk into a vector (using Gemini embeddings).
3. Store these in a searchable vector database (FAISS).
4. For each question, retrieve the top-k most relevant chunks to pass to Gemini for answer generation.

In [ ]:
from langchain.vectorstores import FAISS #Similarity search library
from langchain_google_genai import GoogleGenerativeAIEmbeddings #Text to numerical vectors
from langchain.text_splitter import CharacterTextSplitter #Break long text into overlapping chunks
from langchain.docstore.document import Document #Wraps text to work in vector DB

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", api_key="GOOGLE_API_KEY") #Embedding model
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200) #Split text into chunks



#Split and store in vector DB
docs = splitter.create_documents([paper_text])
vectorstore = FAISS.from_documents(docs, embeddings)

#Search for relevant context
def get_context_with_citations(question, k=3):
    results = vectorstore.similarity_search(question, k=k)
    cited_chunks = []
    for doc in results:
        page = doc.metadata.get("page", "N/A")
        cited_chunks.append(f"(Page {page}) {doc.page_content.strip()}")
    return "\n\n".join(cited_chunks)


In [ ]:
question = "What are the main contributions of this paper?"
context = get_context_with_citations(question)

full_prompt = f"""Use the following context to answer the question:

{context}

Question: {question}
"""

response = model.invoke(full_prompt)
answer = response.content #Clean answer
answer=answer.replace('*','')
print(answer)

In [ ]:
#Summary:
summary_and_toc = summarize_paper(paper_text)
print(summary_and_toc)